In [19]:
import numpy as np 
import pandas as pd

In [20]:
base_path = "data/output/"

In [21]:
nedbit_path = "nedbit_features"
df_nebit_features = pd.read_csv(base_path + nedbit_path, sep=",")
df_nebit_features

,name,class,degree,ring,NetRank,NetShort,HeatDiff,InfoDiff
0,cg00394221_FCRL1,1,4113,0,0.0,0.0,0.0,0.0
1,cg00755661_CTTNBP2NL,1,4077,0,0.0,0.0,0.0,0.0
2,cg01343097_OR2M1P,1,3916,0,0.0,0.0,0.0,0.0
3,cg01833436_SDCCAG8,1,4280,0,0.0,0.0,0.0,0.0
4,cg01833436_AKT3,1,4280,0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...
11751,cg01909024_PAQR7,0,1,0,0.0,0.0,0.0,0.0
11752,cg02073054_IL19,0,1,0,0.0,0.0,0.0,0.0
11753,cg02073054_IL19.1,0,1,0,0.0,0.0,0.0,0.0
11754,cg05262634_CDCA8,0,2,0,0.0,0.0,0.0,0.0


In [4]:
#df_test_probe_genes = pd.read_csv(base_path + "test_probe_genes.csv", sep="\t")
#probe_gene_list = df_test_probe_genes.iloc[:, 0].tolist()
#df_test_probe_genes, len(probe_gene_list), probe_gene_list[:5]

In [5]:
#df_nebit_features_test = df_nebit_features[df_nebit_features["name"].isin(probe_gene_list)]
#df_nebit_features_test.reset_index(drop=True, inplace=True)
#df_nebit_features_test

In [6]:
nebit_features = df_nebit_features.iloc[:, 3:]
nebit_features

,ring,NetRank,NetShort,HeatDiff,InfoDiff
0,1,0.905257,2.281636e+03,0.352461,0.227665
1,1,0.894339,2.250929e+03,0.410495,0.227579
2,1,0.676167,2.697290e+03,1.546465,0.227793
3,1,0.612830,2.922246e+03,1.893043,0.227296
4,1,0.612830,2.922246e+03,1.893043,0.227296
...,...,...,...,...,...
11751,3,2.982511,1.573376e+02,0.000007,0.000155
11752,0,0.000000,6.538936e-305,0.000000,0.000000
11753,0,0.000000,6.538936e-305,0.000000,0.000000
11754,3,2.987049,1.558967e+02,0.000007,0.000422


In [7]:
# normalise NetShort feature

from sklearn.preprocessing import normalize, RobustScaler

netshort = np.array(nebit_features["NetShort"].tolist())
netshort = netshort.reshape(-1, 1)
print(len(netshort), netshort.shape)
transformer = RobustScaler().fit(netshort)
#norm_netshort = normalize(netshort, norm="l2")
norm_netshort = transformer.transform(netshort)
norm_netshort

11756 (11756, 1)


array([[ 0.81468116],
       [ 0.77192887],
       [ 1.39338173],
       ...,
       [-2.36196109],
       [-2.14491155],
       [-2.13550694]])

In [8]:
nebit_features["NetShort"] = norm_netshort
nebit_features

,ring,NetRank,NetShort,HeatDiff,InfoDiff
0,1,0.905257,0.814681,0.352461,0.227665
1,1,0.894339,0.771929,0.410495,0.227579
2,1,0.676167,1.393382,1.546465,0.227793
3,1,0.612830,1.706580,1.893043,0.227296
4,1,0.612830,1.706580,1.893043,0.227296
...,...,...,...,...,...
11751,3,2.982511,-2.142905,0.000007,0.000155
11752,0,0.000000,-2.361961,0.000000,0.000000
11753,0,0.000000,-2.361961,0.000000,0.000000
11754,3,2.987049,-2.144912,0.000007,0.000422


In [9]:
output_gene_ranking_path = "output_gene_ranking_only_positive_corr"
df_apu_labels = pd.read_csv(base_path + output_gene_ranking_path, sep=" ", header=None)
df_apu_labels

,0,1,2
0,cg00394221_FCRL1,0.832421,1
1,cg00755661_CTTNBP2NL,0.831663,1
2,cg01343097_OR2M1P,0.835457,1
3,cg01833436_SDCCAG8,0.834424,1
4,cg01833436_AKT3,0.834424,1
...,...,...,...
11751,cg01909024_PAQR7,-0.552101,5
11752,cg02073054_IL19,0.810015,1
11753,cg02073054_IL19.1,0.810015,1
11754,cg05262634_CDCA8,-0.552093,5


In [10]:
l_name = list()
l_labels = list()
for i, item in df_nebit_features.iterrows():
    r_val = item.values
    matched_row = df_apu_labels[df_apu_labels.loc[:, 0] == r_val[0]]
    if len(matched_row.index) > 0:
        l_name.append(r_val[0])
        l_labels.append(matched_row.values[0][2])

df_labels = pd.DataFrame(zip(l_name, l_labels), columns=["feature_name", "labels"])
df_labels

,feature_name,labels
0,cg00394221_FCRL1,1
1,cg00755661_CTTNBP2NL,1
2,cg01343097_OR2M1P,1
3,cg01833436_SDCCAG8,1
4,cg01833436_AKT3,1
...,...,...
11751,cg01909024_PAQR7,5
11752,cg02073054_IL19,1
11753,cg02073054_IL19.1,1
11754,cg05262634_CDCA8,5


In [11]:
labels = df_labels["labels"].tolist()

In [12]:
#import umap

#n_neighbors=35 #10 #5
#min_dist=0.999 #0.99 #0.99 #0.3
#metric='correlation'

#embeddings = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist, metric='correlation').fit_transform(nebit_features)

In [13]:
#import seaborn as sns
#import matplotlib.pyplot as plt

# Create a DataFrame with UMAP components and labels
#data = {"UMAP1": embeddings[:, 0], "UMAP2": embeddings[:, 1], "Label": labels}
#df = pd.DataFrame(data)

#plt.figure(figsize=(8, 6))
#sns.scatterplot(x="UMAP1", y="UMAP2", hue="Label", data=df, palette="viridis", s=50, alpha=1.0)
#, palette="viridis", s=50, alpha=0.9
#plt.title("UMAP Visualization of NeDBIT Embeddings")
#plt.savefig(base_path + "umap_NeDBIT_features_{}_{}.pdf".format(n_neighbors, min_dist))
#plt.show()

In [ ]:
df_merged_signals = pd.read_csv("../nanodiag_datasets/GSE175758/merged_signals.csv", sep="\t", engine="c")
df_merged_signals

In [ ]:
feature_names = df_labels["feature_name"].tolist()

In [ ]:
dnam_signals = df_merged_signals[feature_names]
dnam_signals

In [ ]:
dnam_signals_transpose = dnam_signals.transpose()
dnam_signals_transpose

In [ ]:
dnam_signals_transpose.to_csv(base_path + "dnam_signals_transpose.csv")

In [ ]:
dnam_signals_transpose = dnam_signals_transpose.reset_index()
dnam_signals_transpose

In [ ]:
dnam_features = dnam_signals_transpose.iloc[:, 1:]
dnam_features

In [ ]:
#n_neighbors=35 #10 #10 #5
#min_dist=0.999 #0.99 #0.99 #0.3
#metric='correlation'

#dnam_embeddings = umap.UMAP(n_neighbors=n_neighbors,
#                       min_dist=min_dist,
#                       metric='correlation').fit_transform(dnam_features)

In [ ]:
#import seaborn as sns
#import matplotlib.pyplot as plt

# Create a DataFrame with UMAP components and labels
#data = {"UMAP1": dnam_embeddings[:, 0], "UMAP2": dnam_embeddings[:, 1], "Label": labels}
#df = pd.DataFrame(data)

#plt.figure(figsize=(8, 6))
#sns.scatterplot(x="UMAP1", y="UMAP2", hue="Label", data=df, palette="viridis", s=50, alpha=0.9) #, palette="viridis", s=50, alpha=0.9
#plt.title("UMAP Visualization of Methylation patterns embeddings")
#plt.savefig(base_path + "umap_dnam_features_{}_{}.pdf".format(n_neighbors, min_dist))
#plt.show()

In [ ]:
#nebit_features = nebit_features.reset_index()
nebit_features

In [ ]:
dnam_features

In [ ]:
#nebit_dnam_features = pd.concat([nebit_features, dnam_signals_transpose.iloc[:, 1:]], axis=1)
nebit_dnam_features = pd.concat([nebit_features, dnam_features], axis=1)
nebit_dnam_features

In [ ]:
#n_neighbors=20 #20 #10 #5
#min_dist=0.999 #0.8 #0.99 #0.3
#metric='correlation'

#nebit_dnam_embeddings = umap.UMAP(n_neighbors=n_neighbors,
#                       min_dist=min_dist,
#                       metric='correlation').fit_transform(nebit_dnam_features)

In [ ]:
#import seaborn as sns
#import matplotlib.pyplot as plt

# Create a data frame with UMAP components and labels
#data = {"UMAP1": nebit_dnam_embeddings[:, 0], "UMAP2": nebit_dnam_embeddings[:, 1], "Label": labels}
#df = pd.DataFrame(data)

#plt.figure(figsize=(8, 6))
#sns.scatterplot(x="UMAP1", y="UMAP2", hue="Label", data=df, palette="viridis", s=50, alpha=0.9) #, palette="viridis", s=50, alpha=0.9
#plt.title("UMAP Visualization of NeDBIT+Methylation embeddings")
#plt.savefig(base_path + "umap_NeDBIT_dnam_features_{}_{}.pdf".format(n_neighbors, min_dist))
#plt.show()

In [ ]:
df_nebit_dnam_features = nebit_dnam_features

In [ ]:
df_nebit_dnam_features["labels"] = labels
#df_nebit_dnam_features["feature_names"] = probe_gene_list

In [ ]:
file_path = base_path + "df_nebit_dnam_features.csv"
df_nebit_dnam_features.to_csv(file_path, sep="\t", header=None, index=None)
df_nebit_dnam_features

In [ ]:
#feature_gene_names = dnam_signals_transpose.iloc[0:, 0]
#feature_gene_names

In [ ]:
#df_feature_names = pd.DataFrame(feature_gene_names)
#file_path = base_path + "df_feature_names.csv"
#df_feature_names.to_csv(file_path, sep="\t", index=None)
#df_feature_names

In [ ]:
import torch
import seaborn as sns
import matplotlib.pyplot as plt
import umap

base_path = "naipu_processed_data/only_positive_corr_data/"
base_plot_path = "plots/positive_negative_corr_data/"
embed_conv = torch.load(base_path + "embed_conv.pt")
embed_batch_norm = torch.load(base_path + "embed_batch_norm.pt")
true_labels = torch.load(base_path + "true_labels.pt")
true_labels = [int(item) + 1 for item in true_labels]

In [ ]:
n_neighbors=10 #10 #5
min_dist=0.99 #0.99 #0.3
metric='correlation'

umap_bn_embed = umap.UMAP(n_neighbors=n_neighbors,
                       min_dist=min_dist,
                       metric='correlation').fit_transform(embed_batch_norm)

# Create a DataFrame with UMAP components and labels
data = {"UMAP1": umap_bn_embed[:, 0], "UMAP2": umap_bn_embed[:, 1], "Label": true_labels}
df = pd.DataFrame(data)

plt.figure(figsize=(8, 6))
sns.scatterplot(x="UMAP1", y="UMAP2", hue="Label", data=df, palette="viridis", s=50, alpha=0.9)
plt.title("UMAP Visualization of node embeddings (GNN)")
plt.savefig(base_plot_path + "umap_batch_norm_{}_{}.pdf".format(n_neighbors, min_dist))
plt.show()

In [ ]:
n_neighbors=10 #5
min_dist=0.99 #0.3
metric='correlation'

umap_conv_embed = umap.UMAP(n_neighbors=n_neighbors,
                       min_dist=min_dist,
                       metric='correlation').fit_transform(embed_conv)

# Create a DataFrame with UMAP components and labels
data = {"UMAP1": umap_conv_embed[:, 0], "UMAP2": umap_conv_embed[:, 1], "Label": true_labels}
df = pd.DataFrame(data)

plt.figure(figsize=(8, 6))
sns.scatterplot(x="UMAP1", y="UMAP2", hue="Label", data=df, palette="viridis", s=50, alpha=0.9) #, palette="viridis", s=50, alpha=0.9
plt.title("UMAP Visualization of node embeddings (GNN)")
plt.savefig(base_plot_path + "umap_conv_{}_{}.pdf".format(n_neighbors, min_dist))
plt.show()

In [ ]:
#df_output_gene_rankings = pd.read_csv(base_path + "output_gene_ranking_only_positive_corr", sep="\t", header=None)
#df_output_gene_rankings = df_output_gene_rankings.sort_values(by=[0])
#df_output_gene_rankings.to_csv(base_path + "output_gene_ranking_only_positive_corr_sorted", index=None)
#df_output_gene_rankings

In [ ]:
'''probes = dict()
for i, row in df_output_gene_rankings.iterrows():
    r_values = row.values[0].split(" ")
    name = r_values[0]
    p_name = name.split("_")[0]
    if p_name not in probes:
        probes[p_name] = 1
    else:
        probes[p_name] += 1'''

In [ ]:
#dict(sorted(probes.items(), key=lambda item: item[1], reverse=True))

In [ ]:
import pandas as pd
import numpy as np

path = "naipu_processed_data/only_positive_corr_data/"
df_test_data = pd.read_csv(path + "df_nebit_dnam_features.csv", sep="\t", header=None)
df_test_data

In [ ]:
df_f_name_labels = df_test_data.loc[:, 39:]
df_f_name_labels

In [ ]:
df_seeds = pd.read_csv(path + "seed_features.tsv", sep="\t", header=None)
df_seeds